# LLM-анализ блока «Мотивация клиента на оплату» в диалогах взыскания

## Постановка задачи

Нужно разработать решение на основе prompt'ов для LLM, которое по тексту диалога оператора с клиентом и сроку просрочки определяет, выполнен ли блок **«Мотивация клиента на оплату»**.

Оценка выполняется по правилам из ТЗ: учитываются категория клиента, согласие или отказ от оплаты, озвученные последствия, допустимость формулировок и исключения.

## Категории и статус клиента

Категория клиента определяется по сроку просрочки задолженности.

| Категория | Срок просрочки |
|---|---:|
| A | 1–30 дней |
| Б | 31–45 дней |

Статус клиента определяется по его реакции на предложение оператора об оплате.

| Статус клиента | Описание |
|---|---|
| Согласен оплатить | Клиент согласен оплатить в предложенный оператором срок |
| Не согласен оплатить | Клиент отказывается, торгуется или просит отсрочку |

Короткий ответ вроде «угу» может считаться согласием, если он дан как подтверждение оплаты в ответ на вопрос оператора.

## Правила оценки по категориям

| Категория | Статус клиента | Условие выполнения критерия |
|---|---|---|
| A | Согласен | 1 последствие неоплаты, констатация действующего последствия или позитивное уведомление об отмене последствий |
| A | Не согласен | 2 последствия неоплаты или констатация действующих последствий |
| Б | Согласен | 1 последствие из категории A |
| Б | Не согласен | 1 последствие из категории A + 1 последствие из категории Б |

## Последствия

| Категория A | Категория Б, дополнительный список |
|---|---|
| ухудшение кредитной истории | требование полного досрочного погашения |
| штрафы / неустойки | ст. 811 ГК РФ |
| звонки | продажа долга |
| выезд сотрудника | передача в работу коллекторов |
| продажа долга |  |
| передача в работу коллекторов |  |

Для клиентов категории A ошибкой периода считается использование последствий, применимых только к категории Б, например требования полного досрочного погашения или ссылки на ст. 811 ГК РФ.

## Допустимые и запрещённые формулировки

Оператор должен использовать допустимые формулировки: **«вправе»**, **«может»**, **«возможно»**.

| Разрешено | Запрещено |
|---|---|
| вероятностные формулировки: «может», «вправе», «возможно» | гарантии наступления последствий: «будет», «передадим» |
| точные суммы | точные даты будущих последствий |
| констатация уже начисляемых штрафов | формулировки вида «завтра передадим» |
| констатация ухудшающейся кредитной истории |  |
| констатация поступающих звонков |  |

## Исключения

Критерий считается выполненным автоматически, если в диалоге упоминается одно из обстоятельств:

- банкротство;
- ЧС;
- военные действия;
- тюрьма;
- армия;
- мошенничество;
- смерть плательщика;
- инвалидность;
- лечение.

Также критерий считается выполненным автоматически, если оператор не попрощался в конце диалога.

## Примеры из ТЗ

| Кейс | Ситуация | Ожидаемый результат | Причина |
|---|---|---|---|
| Согласие + позитивное уведомление | Просрочка 10 дней, клиент: «Сегодня оплачу». Оператор: «Как только платеж поступит, звонки прекратятся» | Выполнено | Есть позитивное уведомление об отмене последствия |
| Нет мотивации | Клиент согласен. Оператор: «Ждём платежа, себя не подводите» | Не выполнено | Не озвучены последствия |
| Ошибка периода | Просрочка 10 дней, оператор говорит про полное досрочное погашение и передачу долга коллекторам | Не выполнено | Для категории A использовано последствие категории Б |
| Несогласие + 2 последствия | Просрочка 22 дня, клиент не может оплатить сегодня, оператор говорит про штрафы и звонки | Выполнено | Озвучены 2 последствия |
| Запрещённая форма | «Если не оплатите до 13 апреля, мы передадим дело коллекторам» | Не выполнено | Есть точная дата и гарантия |
| Исключение | Клиент сообщает о лечении | Выполнено | Сработало исключение |
| «Угу» как согласие | Клиент отвечает «угу» на подтверждение оплаты | Выполнено | Ответ засчитан как согласие |

## Целевые признаки для оценки

| Признак | Зачем нужен |
|---|---|
| Срок просрочки | Определение категории клиента |
| Статус клиента | Выбор правила для согласного / несогласного клиента |
| Последствия, озвученные оператором | Проверка наличия мотивации |
| Категория последствия | Проверка соответствия категории клиента |
| Формулировка последствия | Проверка запрета гарантий и точных дат |
| Исключения | Автоматическое выполнение критерия |
| Завершение диалога | Проверка признака прерванного диалога |

## Baseline-подход

В baseline используется один монолитный prompt и один вызов LLM:

```text
текст диалога + срок просрочки → prompt → JSON-оценка
```

Внутри prompt модель должна последовательно:
1. проверить исключения;
2. определить категорию клиента;
3. определить статус клиента;
4. найти последствия;
5. проверить допустимость формулировок;
6. сопоставить признаки с правилами категории;
7. вернуть итоговое решение.

Такой подход выбран как стартовый: он проще в реализации, быстрее проверяется и достаточен для первичной оценки качества prompt'а.

| Плюсы | Ограничения |
|---|---|
| Один вызов модели | Сложнее локализовать ошибку внутри reasoning |
| Простая реализация | Большой prompt может быть менее устойчив на сложных кейсах |
| Удобно быстро тестировать | Часть проверок потенциально можно вынести в детерминированную логику |

## Допущения baseline

- Prompt'ы написаны на русском языке, так как диалоги и правила из ТЗ русскоязычные.
- Для некоторых multilingual-моделей английские system-инструкции могут быть стабильнее; это можно проверить отдельно при адаптации решения под конкретную LLM.
- Целевая LLM заранее неизвестна, поэтому в production prompt strategy нужно подбирать под конкретную модель.
- Качество оценки зависит от качества текстовой расшифровки диалога.
- Спорные случаи могут требовать ручной проверки или дополнительной валидации.

# Реализация baseline

In [ ]:
import json
from typing import Any, Dict, List, Optional

## System prompt

Ниже задаётся основной prompt с правилами оценки из ТЗ.

In [ ]:
SYSTEM_PROMPT = """
Ты оцениваешь качество мотивации клиента на оплату задолженности.
Следуй инструкции пользователя.
Верни только валидный JSON.
""".strip()

## User prompt template

В user prompt передаются только переменные конкретного кейса: срок просрочки и текст диалога.

In [ ]:
overdue_for_A = "категория A: 1–30 дней просрочки"
overdue_for_B = "категория Б: 31–45 дней просрочки"

payment_true = """
клиент согласен оплатить задолженность.
Согласие может быть выражено явно: «оплачу», «внесу платёж», «закрою сегодня», «да, подтверждаю».
Оператор может не называть конкретный срок, если клиент сам заявил намерение оплатить.
""".strip()

payment_false = """
клиент отказывается, торгуется, просит отсрочку, говорит «не могу», «постараюсь», «возможно»,
или не подтверждает оплату уверенно.
""".strip()

In [ ]:
eval_rules_A = """
- если клиент согласен оплатить: должно быть озвучено 1 последствие неоплаты,
  констатация действующего последствия или позитивное уведомление об отмене последствий;
- если клиент не согласен оплатить: должно быть озвучено 2 последствия неоплаты
  или констатация действующих последствий.
""".strip()

eval_rules_B = """
- если клиент согласен оплатить: должно быть озвучено 1 последствие именно из категории A;
- если клиент не согласен оплатить: должно быть озвучено минимум 1 последствие из категории A
  и минимум 1 дополнительное последствие из категории Б.

Важно:
- для согласного клиента категории Б недостаточно озвучить только дополнительное последствие категории Б;
- дополнительные последствия категории Б используются как обязательное дополнение только если клиент не согласен оплатить.
""".strip()

In [ ]:
motivation_forms = """
Мотивацией считается одно из трёх:

1. Последствие неоплаты:
   например штрафы / неустойки, звонки, ухудшение кредитной истории, выезд сотрудника,
   продажа долга, передача в работу коллекторов, требование полного досрочного погашения,
   ст. 811 ГК РФ.

2. Констатация действующего последствия:
   например уже начисляются штрафы, уже поступают звонки, уже ухудшается кредитная история.

3. Позитивное уведомление об отмене последствия после оплаты:
   например после оплаты звонки прекратятся,
   после оплаты информация по кредитной истории обновится,
   после поступления платежа взаимодействие по задолженности будет прекращено.

Позитивное уведомление об отмене последствия после оплаты засчитывается как мотивация.
""".strip()

In [ ]:
consequences_A = """
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.
""".strip()

consequences_B = """
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.
""".strip()

In [ ]:
allowed_words = """
- вправе;
- может;
- возможно.
""".strip()

not_allowed_words = """
- точные даты наступления последствий;
- гарантии будущих последствий, например: «будет», «завтра передадим», «мы передадим»;
- утвердительные формулировки будущих последствий.
""".strip()

In [ ]:
exceptional_situations = """
- банкротство;
- ЧС;
- военные действия;
- тюрьма;
- армия;
- мошенничество;
- смерть плательщика;
- инвалидность;
- лечение.
""".strip()

auto_pass_rules = """
Критерий считается выполненным автоматически, если:
- в диалоге найдено исключение;
- оператор не попрощался в конце диалога.
""".strip()

In [ ]:
role_rules = """
- последствия и нарушения ищи только в репликах оператора;
- согласие или отказ клиента определяй только по репликам клиента;
- не засчитывай последствия, которые озвучил клиент;
- не придумывай последствия, которых нет в репликах оператора;
- не используй списки правил как найденные последствия в диалоге.
""".strip()

In [ ]:
fail_rules = """
Критерий считается невыполненным, если:
- оператор не озвучил нужное количество последствий для категории клиента и статуса клиента;
- оператор использовал запрещённую формулировку;
- оператор назвал точную дату наступления последствия;
- оператор гарантировал будущее последствие;
- клиент относится к категории A, а оператор озвучил требование полного досрочного погашения или ст. 811 ГК РФ.
""".strip()

In [ ]:
restrictions = """
- оценивай только блок «Мотивация клиента на оплату»;
- не оценивай другие аспекты диалога;
- учитывай только текст диалога и срок просрочки;
- не придумывай факты, последствия, согласие клиента или нарушения;
- ответ «угу» может считаться согласием, если он подтверждает оплату;
- поле detected_consequences заполняй только по явно найденным репликам оператора;
- если последствий нет, detected_consequences должен быть пустым списком [];
- если оператор сказал, что после оплаты последствие прекратится или ситуация улучшится, засчитывай это как мотивацию;
- позитивное уведомление записывай в detected_consequences с type="позитивное уведомление".
""".strip()

In [ ]:
decision_steps = """
Порядок проверки:

1. Определи категорию клиента только по сроку просрочки.

2. Проверь автоматический зачёт:
   - исключение;
   - оператор не попрощался в конце диалога.

3. Определи статус клиента:
   - согласен оплатить;
   - не согласен оплатить.

4. Найди последствия, озвученные оператором.

5. Найди нарушения в репликах оператора.

6. Прими решение:
   - сначала проверь автоматический зачёт;
   - затем проверь нарушения;
   - затем проверь, хватает ли последствий по категории клиента и статусу клиента.
""".strip()

## Формат ответа модели

Модель должна возвращать валидный JSON, чтобы результат можно было автоматически проверить и сохранить.

```json
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [
    {
      "text": "После оплаты звонки прекратятся",
      "type": "звонки",
      "category": "A"
    }
  ],
  "violations": [],
  "explanation": "Клиент согласился оплатить, оператор озвучил позитивное уведомление об отмене последствия."
}
```

### Основные поля ответа

| Поле | Назначение |
|---|---|
| criterion_met | Итоговый результат проверки |
| decision | Текстовый итог проверки |
| client_category | Категория клиента |
| client_status | Статус клиента |
| exception_detected | Было ли найдено исключение |
| interrupted_dialog | Был ли диалог прерван |
| detected_consequences | Найденные последствия |
| violations | Найденные нарушения |
| explanation | Объяснение итогового решения |

In [ ]:
output_schema = """
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [],
  "violations": [],
  "explanation": "Краткое объяснение в 1 предложении без прямых цитат из диалога."
}
""".strip()

In [ ]:
def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени, насколько качественно оператор мотивирует клиента на оплату задолженности.

Входные данные:
Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории клиентов:
- {overdue_for_A}
- {overdue_for_B}

Как определить согласие клиента:
Согласен:
{payment_true}

Не согласен:
{payment_false}

Правила оценки категории A:
{eval_rules_A}

Правила оценки категории Б:
{eval_rules_B}

Последствия категории A:
{consequences_A}

Дополнительные последствия категории Б:
{consequences_B}

Формы мотивации:
{motivation_forms}

Допустимые формулировки оператора:
{allowed_words}

Недопустимые формулировки оператора:
{not_allowed_words}

Исключения:
{exceptional_situations}

Правила автоматического зачёта:
{auto_pass_rules}

Правила ролей:
{role_rules}

Правила незачёта:
{fail_rules}

Общие ограничения:
{restrictions}

Порядок проверки:
{decision_steps}

Верни только валидный JSON без markdown-разметки и дополнительных комментариев.

Формат ответа:
{output_schema}
""".strip()

## Вызов LLM

In [ ]:
from openai import OpenAI, APITimeoutError, APIConnectionError, RateLimitError, APIStatusError

In [ ]:
import os
import time

In [ ]:
import getpass
API_KEY = getpass.getpass("Enter API key: ")
BASE_URL = getpass.getpass("Enter base URL: ")

Enter API key: ··········
Enter base URL: ··········


In [ ]:
MODEL_NAME = "gpt-4o-mini"

In [ ]:
client = OpenAI(api_key=API_KEY,
                base_url=BASE_URL,
                timeout=15.0,
                max_retries=3,)

In [ ]:
def evaluate_dialogue(
    dialogue: str,
    overdue_days: int,
    model: str = MODEL_NAME,
    temperature: float = 0.0,
    max_tokens: int = 700,
    max_attempts: int = 3,
    verbose: bool = True,
) -> Dict[str, Any]:
    user_prompt = build_user_prompt(
        dialogue=dialogue,
        overdue_days=overdue_days,
    )

    start_time = time.perf_counter()
    last_error = None
    raw_content = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )

            raw_content = response.choices[0].message.content
            parsed_result = json.loads(raw_content)

            elapsed_time = time.perf_counter() - start_time

            if verbose:
                print("Model:", model)
                print("Base URL:", BASE_URL)
                print("Prompt length:", len(user_prompt))
                print(f"Attempts used: {attempt}")
                print(f"Elapsed time: {elapsed_time:.2f} sec")
                print("Raw answer:\n", raw_content)

            parsed_result["_elapsed_time_sec"] = round(elapsed_time, 2)
            parsed_result["_prompt_length"] = len(user_prompt)
            parsed_result["_model"] = model
            parsed_result["_attempts_used"] = attempt

            return parsed_result

        except Exception as error:
            last_error = error

            if verbose:
                print(f"Attempt {attempt}/{max_attempts} failed: {type(error).__name__}")

    raise RuntimeError(
        f"Не удалось получить корректный ответ после {max_attempts} попыток.\n"
        f"Последняя ошибка: {type(last_error).__name__}: {last_error}\n\n"
        f"Последний raw_content:\n{raw_content}"
    )

In [ ]:
def print_evaluation_result(result: Dict[str, Any]) -> None:
    print("=== Результат оценки ===")
    print(f"Итог: {result.get('decision')}")
    print(f"Критерий выполнен: {result.get('criterion_met')}")
    print(f"Категория клиента: {result.get('client_category')}")
    print(f"Статус клиента: {result.get('client_status')}")
    print(f"Исключение найдено: {result.get('exception_detected')}")
    print(f"Прерванный диалог: {result.get('interrupted_dialog')}")

    print("\n=== Найденные последствия ===")
    consequences = result.get("detected_consequences", [])
    if consequences:
        for idx, consequence in enumerate(consequences, start=1):
            print(f"{idx}. Текст: {consequence.get('text')}")
            print(f"   Тип: {consequence.get('type')}")
            print(f"   Категория: {consequence.get('category')}")
    else:
        print("Не найдены")

    print("\n=== Нарушения ===")
    violations = result.get("violations", [])
    if violations:
        for idx, violation in enumerate(violations, start=1):
            if isinstance(violation, dict):
                print(f"{idx}. Текст: {violation.get('text')}")
                print(f"   Тип: {violation.get('type')}")
                print(f"   Причина: {violation.get('reason')}")
            else:
                print(f"{idx}. {violation}")
    else:
        print("Не найдены")

    print("\n=== Объяснение ===")
    print(result.get("explanation"))

# Тестирование baseline

Для проверки baseline используется небольшой набор кейсов, покрывающих основные правила из ТЗ: успешные сценарии, отсутствие мотивации, ошибку периода, запрещённые формулировки, исключения и неявное согласие.

Для каждого кейса сравнивается ожидаемый результат с ответом модели.

## Первый тестовый диалог

In [ ]:
test_dialogue = """
Оператор: Подтверждаете оплату до пятницы?
Клиент: Угу.
Оператор: После оплаты звонки прекратятся.
Оператор: До свидания.
""".strip()

In [ ]:
result = evaluate_dialogue(
    dialogue=test_dialogue,
    overdue_days=10,
)

Model: gpt-4o-mini
Base URL: https://api.artemox.com/v1
Prompt length: 5575
Attempts used: 1
Elapsed time: 12.88 sec
Raw answer:
 {
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [
    {
      "type": "позитивное уведомление",
      "text": "После оплаты звонки прекратятся."
    }
    ],
  "violations": [],
  "explanation": "Клиент выразил согласие оплатить, оператор упомянул позитивное уведомление об отмене последствий после оплаты, что соответствует требованиям для категории A."
}


In [ ]:
print_evaluation_result(result)

=== Результат оценки ===
Итог: выполнено
Критерий выполнен: True
Категория клиента: A
Статус клиента: согласен оплатить
Исключение найдено: False
Прерванный диалог: False

=== Найденные последствия ===
1. Текст: После оплаты звонки прекратятся.
   Тип: позитивное уведомление
   Категория: None

=== Нарушения ===
Не найдены

=== Объяснение ===
Клиент выразил согласие оплатить, оператор упомянул позитивное уведомление об отмене последствий после оплаты, что соответствует требованиям для категории A.


## Несколько тестов

### Тестовые примеры

In [ ]:
test_cases_tz = [
    {
        "case_id": "tz_01_A_agree_positive_notification",
        "description": "Пример 1 из ТЗ: категория A, клиент согласен, позитивное уведомление",
        "overdue_days": 10,
        "expected": True,
        "dialogue": """
Клиент: Сегодня оплачу.
Оператор: Как только платеж поступит, звонки прекратятся.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_02_A_agree_no_motivation",
        "description": "Пример 2 из ТЗ: категория A, клиент согласен, нет мотивации",
        "overdue_days": 10,
        "expected": False,
        "dialogue": """
Клиент: Сегодня оплачу.
Оператор: Ждём платежа, себя не подводите.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_03_A_wrong_period",
        "description": "Пример 3 из ТЗ: категория A, ошибка периода",
        "overdue_days": 10,
        "expected": False,
        "dialogue": """
Оператор: Банк может потребовать полное досрочное погашение и передать долг коллекторам.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_04_A_disagree_two_consequences",
        "description": "Пример 4 из ТЗ: категория A, клиент не согласен, 2 последствия",
        "overdue_days": 22,
        "expected": True,
        "dialogue": """
Клиент: Не могу сегодня.
Оператор: Банк вправе ежедневно начислять штрафы — это переплата, долг растёт.
Оператор: Также могут быть звонки вам и контактным лицам, создавая неудобства.
Оператор: Как планируете решать?
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_05_A_forbidden_assertive_wording",
        "description": "Пример 5 из ТЗ: точная дата последствия и гарантия",
        "overdue_days": 15,
        "expected": False,
        "dialogue": """
Оператор: Если не оплатите до 13 апреля, мы передадим дело коллекторам.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_06_exception_treatment",
        "description": "Пример 6 из ТЗ: исключение — лечение",
        "overdue_days": 18,
        "expected": True,
        "dialogue": """
Клиент: У меня инсульт, я на лечении.
Оператор: Понимаю, выздоравливайте.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "tz_08_implicit_agreement_ugu",
        "description": "Пример 8 из ТЗ: «угу» как согласие",
        "overdue_days": 10,
        "expected": True,
        "dialogue": """
Оператор: Подтверждаете оплату до пятницы?
Клиент: Угу.
Оператор: Отлично, после оплаты обновим кредитную историю.
Оператор: До свидания.
""".strip(),
    },
]

In [ ]:
manual_case_tz_words = {
    "case_id": "manual_01_tz_words_complex",
    "description": "Ручной длинный кейс: формулировки близки к ТЗ, проверка нескольких признаков",
    "overdue_days": 22,
    "expected": True,
    "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность, нужно обсудить оплату.
Клиент: Сегодня не могу оплатить, денег нет.
Оператор: Правильно понимаю, что сегодня оплатить не получится?
Клиент: Да, сегодня не смогу.
Оператор: Банк вправе ежедневно начислять штрафы — это переплата, долг растёт.
Оператор: Также могут быть звонки вам и контактным лицам, создавая неудобства.
Оператор: Как планируете решать вопрос с оплатой?
Клиент: Постараюсь внести платёж завтра или послезавтра.
Оператор: Хорошо, ожидаем оплату. До свидания.
""".strip(),
}

In [ ]:
manual_case_synonyms = {
    "case_id": "manual_02_synonyms_complex",
    "description": "Ручной длинный кейс: естественные синонимы вместо точных формулировок",
    "overdue_days": 36,
    "expected": True,
    "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченной задолженности. Когда сможете внести платёж?
Клиент: Сейчас не готов платить, нужна отсрочка хотя бы на пару недель.
Оператор: То есть в предложенный срок оплату подтвердить не можете?
Клиент: Да, пока не могу.
Оператор: В таком случае банк может продолжить начисление неустойки, и сумма задолженности станет больше.
Оператор: Также банк вправе потребовать вернуть всю сумму задолженности раньше установленного графика.
Клиент: Понял, но сейчас всё равно денег нет.
Оператор: Какой срок оплаты вы можете назвать сейчас?
Клиент: Попробую решить вопрос в течение недели.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
}

### Второй тест baseline

In [ ]:
test_cases = test_cases_tz + [manual_case_tz_words,
                              manual_case_synonyms,]

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def run_evaluation(test_cases: list[dict],
                   verbose: bool = False,) -> tuple[pd.DataFrame, float]:

    results = []

    total_start_time = time.perf_counter()

    for case in tqdm(test_cases, desc="Evaluating dialogues"):
        result = evaluate_dialogue(dialogue=case["dialogue"],
                                   overdue_days=case["overdue_days"],
                                   verbose=verbose,)

        results.append({"case_id": case["case_id"],
                        "description": case["description"],
                        "overdue_days": case["overdue_days"],
                        "expected": case["expected"],
                        "predicted": result.get("criterion_met"),
                        "is_correct": result.get("criterion_met") == case["expected"],
                        "decision": result.get("decision"),
                        "client_category": result.get("client_category"),
                        "client_status": result.get("client_status"),
                        "exception_detected": result.get("exception_detected"),
                        "interrupted_dialog": result.get("interrupted_dialog"),
                        "detected_consequences": result.get("detected_consequences"),
                        "violations": result.get("violations"),
                        "elapsed_time_sec": result.get("_elapsed_time_sec"),
                        "explanation": result.get("explanation"),})

    total_elapsed_time = time.perf_counter() - total_start_time

    results_df = pd.DataFrame(results)

    return results_df, total_elapsed_time

In [ ]:
results_df, total_elapsed_time = run_evaluation(test_cases=test_cases,
                                                verbose=False,)

results_df

Evaluating dialogues:   0%|          | 0/9 [00:00<?, ?it/s]

,case_id,description,overdue_days,expected,predicted,is_correct,decision,client_category,client_status,exception_detected,interrupted_dialog,detected_consequences,violations,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",10,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'последствие неоплаты', 'text': 'Как...",[],11.66,"Клиент выразил намерение оплатить, оператор по..."
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",10,False,True,False,выполнено,A,согласен оплатить,False,False,[],[],11.87,"Клиент выразил намерение оплатить, а оператор ..."
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",10,False,True,False,выполнено,A,согласен оплатить,False,False,[],[],61.10,Категория A и статус клиента совпадают с согла...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",22,True,True,True,выполнено,A,не согласен оплатить,False,False,[],[],11.84,Срок просрочки 22 дня ставит клиента в категор...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,15,False,True,False,выполнено,A,согласен оплатить,False,False,"[{'type': 'последствие неоплаты', 'text': 'Есл...",[],12.21,Срок просрочки 15 дней относится к категории A...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,18,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[],12.79,Клиенту присвоена категория A; оператор не поп...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,10,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[],12.58,"Клиент согласен оплатить, оператор упомянул по..."
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",22,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[оператор озвучил последствия неоплаты до подт...,28.16,"Клиент заявил намерение оплатить, оператор ука..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,36,True,True,True,выполнено,Б,не согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'statement...",[],11.83,Клиент на 36-й день просрочки относится к кате...


In [ ]:
summary_df = results_df[["case_id","description","expected","predicted","is_correct",
                         "decision","elapsed_time_sec","explanation",]]

summary_df

,case_id,description,expected,predicted,is_correct,decision,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",True,True,True,выполнено,11.66,"Клиент выразил намерение оплатить, оператор по..."
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,True,False,выполнено,11.87,"Клиент выразил намерение оплатить, а оператор ..."
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,False,выполнено,61.10,Категория A и статус клиента совпадают с согла...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,True,True,выполнено,11.84,Срок просрочки 22 дня ставит клиента в категор...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,False,выполнено,12.21,Срок просрочки 15 дней относится к категории A...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,True,True,выполнено,12.79,Клиенту присвоена категория A; оператор не поп...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,True,True,выполнено,12.58,"Клиент согласен оплатить, оператор упомянул по..."
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,True,True,выполнено,28.16,"Клиент заявил намерение оплатить, оператор ука..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,True,True,True,выполнено,11.83,Клиент на 36-й день просрочки относится к кате...


In [ ]:
print("=== Итоги тестирования ===")
print(f"Всего кейсов: {len(results_df)}")
print(f"Корректных ответов: {results_df['is_correct'].sum()}")
print(f"Accuracy: {results_df['is_correct'].mean():.2%}")
print(f"Общее время: {total_elapsed_time:.2f} сек.")
print(f"Среднее время на кейс: {results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {results_df['elapsed_time_sec'].max():.2f} сек.")

=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 6
Accuracy: 66.67%
Общее время: 174.07 сек.
Среднее время на кейс: 19.34 сек.
Минимальное время: 11.66 сек.
Максимальное время: 61.10 сек.


In [ ]:
incorrect_df = results_df[~results_df["is_correct"]]

incorrect_df[["case_id","description","expected","predicted","decision",
              "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,decision,detected_consequences,violations,explanation
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,True,выполнено,[],[],"Клиент выразил намерение оплатить, а оператор ..."
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,выполнено,[],[],Категория A и статус клиента совпадают с согла...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,выполнено,"[{'type': 'последствие неоплаты', 'text': 'Есл...",[],Срок просрочки 15 дней относится к категории A...


In [ ]:
for _, row in incorrect_df.iterrows():
    print("=" * 80)
    print("CASE:", row["case_id"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("\nEXPLANATION:")
    print(row["explanation"])
    print()

CASE: tz_02_A_agree_no_motivation
EXPECTED: False
PREDICTED: True

EXPLANATION:
Клиент выразил намерение оплатить, а оператор не назвал конкретных последствий неоплаты и не упоминал дату, что соответствует категории A при согласии клиента.

CASE: tz_03_A_wrong_period
EXPECTED: False
PREDICTED: True

EXPLANATION:
Категория A и статус клиента совпадают с согласие на оплату, однако оператор не озвучил ни одного последствия из категории A.

CASE: tz_05_A_forbidden_assertive_wording
EXPECTED: False
PREDICTED: True

EXPLANATION:
Срок просрочки 15 дней относится к категории A; оператор упомянул одно последствие неоплаты и сделал уведомление об передаче коллекторской службе, что соответствует требованиям категории A при согласии клиента.



### Анализ первого прогона baseline

Baseline-решение использует один prompt и один вызов LLM для оценки диалога. Такой подход прост в реализации и позволяет быстро получить первичную оценку качества мотивации оператора.

На тестовых примерах baseline показал, что модель в целом понимает структуру задачи, но допускает системные ошибки:

- путает согласие клиента на оплату с качественной мотивацией со стороны оператора;
- может считать критерий выполненным, даже если оператор не озвучил последствие, констатацию действующего последствия или позитивное уведомление;
- не всегда блокирует критерий при ошибке периода;
- не всегда блокирует критерий при запрещённых утвердительных формулировках.

Главная проблема baseline: модель оценивает общий исход диалога, а не строго выполнение блока «Мотивация клиента на оплату» со стороны оператора.

Далее prompt будет доработан точечно: нужно явно разделить согласие клиента и мотивацию оператора.

## Модификация промпта

In [ ]:
evaluation_focus = """
Важно: оценивается не сам факт согласия клиента на оплату, а качество мотивации со стороны оператора.

Клиент может согласиться оплатить, но критерий всё равно будет не выполнен,
если оператор не озвучил требуемую мотивацию.

Мотивация оператора засчитывается только если оператор сделал хотя бы одно из действий:
- озвучил последствие неоплаты;
- констатировал уже действующее последствие;
- дал позитивное уведомление об отмене последствия после оплаты.

Фразы оператора вроде «ждём платёж», «ожидаем оплату», «себя не подводите»,
«не затягивайте» не являются мотивацией.

Если клиент согласен оплатить, это влияет только на количество требуемых последствий,
но само по себе не означает, что оператор выполнил блок мотивации.
""".strip()

In [ ]:
hard_fail_rules = """
Критерий считается невыполненным, если:
- оператор не озвучил нужную мотивацию;
- оператор использовал запрещённую формулировку будущего последствия;
- оператор назвал точную дату наступления последствия;
- клиент относится к категории A, а оператор озвучил требование полного досрочного погашения или ст. 811 ГК РФ.

Если найдено одно из этих условий, criterion_met должен быть false, даже если клиент согласился оплатить.
""".strip()

In [ ]:
def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени, насколько качественно оператор мотивирует клиента на оплату задолженности.

Входные данные:
Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории клиентов:
- {overdue_for_A}
- {overdue_for_B}

Как определить согласие клиента:
Согласен:
{payment_true}

Не согласен:
{payment_false}

Фокус оценки:
{evaluation_focus}

Правила оценки категории A:
{eval_rules_A}

Правила оценки категории Б:
{eval_rules_B}

Последствия категории A:
{consequences_A}

Дополнительные последствия категории Б:
{consequences_B}

Формы мотивации:
{motivation_forms}

Допустимые формулировки оператора:
{allowed_words}

Недопустимые формулировки оператора:
{not_allowed_words}

Исключения:
{exceptional_situations}

Правила автоматического зачёта:
{auto_pass_rules}

Правила ролей:
{role_rules}

Правила незачёта:
{fail_rules}

Правила обязательного незачёта:
{hard_fail_rules}

Общие ограничения:
{restrictions}

Порядок проверки:
{decision_steps}

Верни только валидный JSON без markdown-разметки и дополнительных комментариев.

Формат ответа:
{output_schema}
""".strip()

### Тест модификации

In [ ]:
results_df, total_elapsed_time = run_evaluation(test_cases=test_cases,
                                                verbose=False,)

results_df

Evaluating dialogues:   0%|          | 0/9 [00:00<?, ?it/s]

,case_id,description,overdue_days,expected,predicted,is_correct,decision,client_category,client_status,exception_detected,interrupted_dialog,detected_consequences,violations,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",10,True,True,True,выполнено,A,согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'з...",[],21.01,Оператор дал позитивное уведомление об отмене ...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",10,False,False,True,не выполнено,A,согласен оплатить,False,False,[],[],28.01,Оператор не озвучил ни одного последствия неоп...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",10,False,False,True,незачёт,A,согласен оплатить,False,False,[],[Оператор не озвучил ни одного последствия нео...,45.07,"Клиент согласен оплатить, но оператор не озвуч..."
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",22,True,False,False,недостаточно данных,A,согласен оплатить,False,False,[],[],30.04,В диалоге не зафиксирована явная мотивация опе...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,15,False,False,True,недостаточно данных,A,согласен оплатить,False,False,[],[],12.08,В диалоге оператор не озвучил ни одного послед...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,18,True,False,False,недостаточно данных для автоматического зачёта,A,согласен оплатить,False,False,[],[],12.65,В диалоге оператор не попрощался в конце бесед...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,10,True,False,False,не выполнено,A,согласен оплатить,False,False,[],[],12.06,Оператор не озвучил ни одного обязательного по...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",22,True,False,False,не выполнено,A,соглаcен оплатить,False,False,[],[],12.29,"В диалоге клиент согласен оплатить, но операто..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,36,True,True,True,выполнено,Б,не согласен оплатить,False,False,"[{'type': 'позитивное уведомление', 'text': 'п...",[оператор назвал возможные последствия как мин...,25.50,К клиенту категории Б оператор упомянул послед...


In [ ]:
summary_df = results_df[["case_id","description","expected","predicted","is_correct",
                         "decision","elapsed_time_sec","explanation",]]

summary_df

,case_id,description,expected,predicted,is_correct,decision,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",True,True,True,выполнено,21.01,Оператор дал позитивное уведомление об отмене ...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,False,True,не выполнено,28.01,Оператор не озвучил ни одного последствия неоп...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,False,True,незачёт,45.07,"Клиент согласен оплатить, но оператор не озвуч..."
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,False,False,недостаточно данных,30.04,В диалоге не зафиксирована явная мотивация опе...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,False,True,недостаточно данных,12.08,В диалоге оператор не озвучил ни одного послед...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,False,False,недостаточно данных для автоматического зачёта,12.65,В диалоге оператор не попрощался в конце бесед...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,False,False,не выполнено,12.06,Оператор не озвучил ни одного обязательного по...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,False,False,не выполнено,12.29,"В диалоге клиент согласен оплатить, но операто..."
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,True,True,True,выполнено,25.50,К клиенту категории Б оператор упомянул послед...


In [ ]:
print("=== Итоги тестирования ===")
print(f"Всего кейсов: {len(results_df)}")
print(f"Корректных ответов: {results_df['is_correct'].sum()}")
print(f"Accuracy: {results_df['is_correct'].mean():.2%}")
print(f"Общее время: {total_elapsed_time:.2f} сек.")
print(f"Среднее время на кейс: {results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {results_df['elapsed_time_sec'].max():.2f} сек.")

=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 5
Accuracy: 55.56%
Общее время: 198.73 сек.
Среднее время на кейс: 22.08 сек.
Минимальное время: 12.06 сек.
Максимальное время: 45.07 сек.


In [ ]:
incorrect_df = results_df[~results_df["is_correct"]]

incorrect_df[["case_id","description","expected","predicted","decision",
              "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,decision,detected_consequences,violations,explanation
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,False,недостаточно данных,[],[],В диалоге не зафиксирована явная мотивация опе...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,False,недостаточно данных для автоматического зачёта,[],[],В диалоге оператор не попрощался в конце бесед...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,False,не выполнено,[],[],Оператор не озвучил ни одного обязательного по...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,False,не выполнено,[],[],"В диалоге клиент согласен оплатить, но операто..."


In [ ]:
for _, row in incorrect_df.iterrows():
    print("=" * 80)
    print("CASE:", row["case_id"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("\nEXPLANATION:")
    print(row["explanation"])
    print()

CASE: tz_02_A_agree_no_motivation
EXPECTED: False
PREDICTED: True

EXPLANATION:
Клиент выразил намерение оплатить, а оператор не назвал конкретных последствий неоплаты и не упоминал дату, что соответствует категории A при согласии клиента.

CASE: tz_03_A_wrong_period
EXPECTED: False
PREDICTED: True

EXPLANATION:
Категория A и статус клиента совпадают с согласие на оплату, однако оператор не озвучил ни одного последствия из категории A.

CASE: tz_05_A_forbidden_assertive_wording
EXPECTED: False
PREDICTED: True

EXPLANATION:
Срок просрочки 15 дней относится к категории A; оператор упомянул одно последствие неоплаты и сделал уведомление об передаче коллекторской службе, что соответствует требованиям категории A при согласии клиента.



СТАЛО ХУЖЕ, УПРОЩАЕМ

## Попытка убрать шум

In [ ]:
SYSTEM_PROMPT = """
Ты оцениваешь, выполнен ли блок «Мотивация клиента на оплату» в диалоге оператора взыскания.
Верни только валидный JSON без markdown-разметки и дополнительных комментариев.
""".strip()

In [ ]:
categories = """
Категория A: 1–30 дней просрочки.
Категория Б: 31–45 дней просрочки.
""".strip()

In [ ]:
consequences = """
Последствия категории A:
- ухудшение кредитной истории;
- штрафы / неустойки;
- звонки;
- выезд сотрудника;
- продажа долга;
- передача в работу коллекторов.

Дополнительные последствия категории Б:
- требование полного досрочного погашения;
- ст. 811 ГК РФ;
- продажа долга;
- передача в работу коллекторов.
""".strip()

In [ ]:
exceptions = """
Критерий считается выполненным автоматически, если:
- в диалоге упоминается банкротство, ЧС, военные действия, тюрьма, армия, мошенничество,
  смерть плательщика, инвалидность или лечение;
- оператор не попрощался в конце диалога.
""".strip()

In [ ]:
evaluation_rules = """
Правила оценки:

1. Определи категорию клиента по сроку просрочки.

2. Определи, согласен ли клиент оплатить:
   - согласен: клиент подтверждает оплату или говорит, что оплатит;
   - не согласен: клиент отказывается, торгуется, просит отсрочку или не подтверждает оплату.

3. Для категории A:
   - если клиент согласен, нужна 1 мотивация;
   - если клиент не согласен, нужны 2 мотивации.

4. Для категории Б:
   - если клиент согласен, нужна 1 мотивация из категории A;
   - если клиент не согласен, нужна 1 мотивация из категории A и 1 дополнительная мотивация из категории Б.

5. Мотивацией считается:
   - последствие неоплаты;
   - констатация действующего последствия;
   - позитивное уведомление об отмене последствия после оплаты.

6. Недопустимо:
   - точные даты наступления последствий;
   - гарантии будущих последствий;
   - для категории A: требование полного досрочного погашения или ст. 811 ГК РФ.

7. Последствия и нарушения учитывай только из реплик оператора.
""".strip()

In [ ]:
output_schema = """
{
  "criterion_met": true,
  "decision": "выполнено",
  "client_category": "A",
  "client_status": "согласен оплатить",
  "exception_detected": false,
  "interrupted_dialog": false,
  "detected_consequences": [],
  "violations": [],
  "explanation": "Краткое объяснение в 1 предложении."
}
""".strip()

In [ ]:
def build_user_prompt(dialogue: str, overdue_days: int) -> str:
    return f"""
Оцени мотивацию оператора на оплату задолженности.

Срок просрочки: {overdue_days} дней

Диалог:
{dialogue}

Категории:
{categories}

Последствия:
{consequences}

Исключения:
{exceptions}

Как оценивать:
{evaluation_rules}

Верни ответ строго в формате JSON:
{output_schema}
""".strip()

### Тест модификации

In [ ]:
results_df, total_elapsed_time = run_evaluation(test_cases=test_cases,
                                                verbose=False,)

results_df

Evaluating dialogues:   0%|          | 0/9 [00:00<?, ?it/s]

,case_id,description,overdue_days,expected,predicted,is_correct,decision,client_category,client_status,exception_detected,interrupted_dialog,detected_consequences,violations,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",10,True,True,True,выполнено,A,согласен оплатить,False,False,[],[],12.29,Срок просрочки 10 дней относится к категории A...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",10,False,False,True,не выполнено,A,согласен оплатить,False,False,[],[],6.27,Оператор не указал ни одного последствия неопл...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",10,False,True,False,выполнено,A,согласен оплатить,False,False,[],[],27.77,По сроку просрочки 10 дней клиент относится к ...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",22,True,True,True,выполнено,A,не согласен оплатить,False,False,[звонки],[],12.30,Срок просрочки 22 дня относится к категории A;...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,15,False,True,False,выполнено,A,не согласен оплатить,False,False,[],[],11.83,Категория A; оператор угрожает передать дело к...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,18,True,True,True,выполнено,A,согласен оплатить,True,False,[],[],60.07,Возрастная мотивация на оплату присутствует: к...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,10,True,True,True,выполнено,A,согласен оплатить,False,False,[уточнение: после оплаты обновим кредитную ист...,[],11.39,Срок просрочки 10 дней относится к категория A...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",22,True,True,True,выполнено,A,согласен оплатить,False,False,"[звонки, штрафы / неустойки, переплата, долг р...",[],12.94,Срок просрочки 22 дня (категория A); клиент вы...
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,36,True,True,True,выполнено,A,не согласен оплатить,False,False,[],[],12.54,Срок просрочки 36 дней относится к категории A...


In [ ]:
summary_df = results_df[["case_id","description","expected","predicted","is_correct",
                         "decision","elapsed_time_sec","explanation",]]

summary_df

,case_id,description,expected,predicted,is_correct,decision,elapsed_time_sec,explanation
0,tz_01_A_agree_positive_notification,"Пример 1 из ТЗ: категория A, клиент согласен, ...",True,True,True,выполнено,12.29,Срок просрочки 10 дней относится к категории A...
1,tz_02_A_agree_no_motivation,"Пример 2 из ТЗ: категория A, клиент согласен, ...",False,False,True,не выполнено,6.27,Оператор не указал ни одного последствия неопл...
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,False,выполнено,27.77,По сроку просрочки 10 дней клиент относится к ...
3,tz_04_A_disagree_two_consequences,"Пример 4 из ТЗ: категория A, клиент не согласе...",True,True,True,выполнено,12.30,Срок просрочки 22 дня относится к категории A;...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,False,выполнено,11.83,Категория A; оператор угрожает передать дело к...
5,tz_06_exception_treatment,Пример 6 из ТЗ: исключение — лечение,True,True,True,выполнено,60.07,Возрастная мотивация на оплату присутствует: к...
6,tz_08_implicit_agreement_ugu,Пример 8 из ТЗ: «угу» как согласие,True,True,True,выполнено,11.39,Срок просрочки 10 дней относится к категория A...
7,manual_01_tz_words_complex,"Ручной длинный кейс: формулировки близки к ТЗ,...",True,True,True,выполнено,12.94,Срок просрочки 22 дня (категория A); клиент вы...
8,manual_02_synonyms_complex,Ручной длинный кейс: естественные синонимы вме...,True,True,True,выполнено,12.54,Срок просрочки 36 дней относится к категории A...


In [ ]:
print("=== Итоги тестирования ===")
print(f"Всего кейсов: {len(results_df)}")
print(f"Корректных ответов: {results_df['is_correct'].sum()}")
print(f"Accuracy: {results_df['is_correct'].mean():.2%}")
print(f"Общее время: {total_elapsed_time:.2f} сек.")
print(f"Среднее время на кейс: {results_df['elapsed_time_sec'].mean():.2f} сек.")
print(f"Минимальное время: {results_df['elapsed_time_sec'].min():.2f} сек.")
print(f"Максимальное время: {results_df['elapsed_time_sec'].max():.2f} сек.")

=== Итоги тестирования ===
Всего кейсов: 9
Корректных ответов: 7
Accuracy: 77.78%
Общее время: 167.41 сек.
Среднее время на кейс: 18.60 сек.
Минимальное время: 6.27 сек.
Максимальное время: 60.07 сек.


In [ ]:
incorrect_df = results_df[~results_df["is_correct"]]

incorrect_df[["case_id","description","expected","predicted","decision",
              "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,decision,detected_consequences,violations,explanation
2,tz_03_A_wrong_period,"Пример 3 из ТЗ: категория A, ошибка периода",False,True,выполнено,[],[],По сроку просрочки 10 дней клиент относится к ...
4,tz_05_A_forbidden_assertive_wording,Пример 5 из ТЗ: точная дата последствия и гара...,False,True,выполнено,[],[],Категория A; оператор угрожает передать дело к...


In [ ]:
for _, row in incorrect_df.iterrows():
    print("=" * 80)
    print("CASE:", row["case_id"])
    print("EXPECTED:", row["expected"])
    print("PREDICTED:", row["predicted"])
    print("\nEXPLANATION:")
    print(row["explanation"])
    print()

CASE: tz_03_A_wrong_period
EXPECTED: False
PREDICTED: True

EXPLANATION:
По сроку просрочки 10 дней клиент относится к категории A; оператор не упоминает дополнительные мотивации при согласии клиента на оплату и в диалоге присутствует только сообщение о возможных последствиях, без упоминания конкретной даты или требований.

CASE: tz_05_A_forbidden_assertive_wording
EXPECTED: False
PREDICTED: True

EXPLANATION:
Категория A; оператор угрожает передать дело коллекторам и не прощается, клиент пока не выразил согласие на оплату.



## Финальный тест Baseline

In [ ]:
realistic_test_cases = [
    {
        "case_id": "real_01_A_agree_positive_notification",
        "description": "Категория A, клиент согласен, оператор даёт позитивное уведомление",
        "overdue_days": 7,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Звоню по вопросу просроченного платежа по договору.
Клиент: Да, я видел уведомление, сегодня внесу оплату.
Оператор: Правильно понимаю, что платёж будет сегодня?
Клиент: Да, сегодня вечером.
Оператор: После поступления платежа звонки по задолженности прекратятся.
Клиент: Хорошо, понял.
Оператор: Спасибо за информацию. До свидания.
""".strip(),
    },
    {
        "case_id": "real_02_A_agree_no_motivation",
        "description": "Категория A, клиент согласен, но оператор не мотивирует",
        "overdue_days": 11,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. Напоминаю о просроченной задолженности.
Клиент: Да, я в курсе, сегодня постараюсь оплатить.
Оператор: Хорошо, тогда ожидаем платёж. Пожалуйста, не затягивайте.
Клиент: Хорошо.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_03_A_disagree_two_A_consequences",
        "description": "Категория A, клиент не согласен, оператор называет два последствия A",
        "overdue_days": 23,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. У вас сохраняется просроченная задолженность. Когда сможете оплатить?
Клиент: Сейчас не могу, денег нет, возможно только через неделю.
Оператор: Понимаю. При отсутствии оплаты банк вправе начислять неустойку, из-за чего сумма долга может увеличиваться.
Оператор: Также могут продолжаться звонки вам и контактным лицам по вопросу задолженности.
Клиент: Понял, но раньше оплатить не получится.
Оператор: Какой срок оплаты можете назвать сейчас?
Клиент: Попробую решить вопрос до конца недели.
Оператор: Спасибо, информацию зафиксировал. До свидания.
""".strip(),
    },
    {
        "case_id": "real_04_A_wrong_period_full_repayment",
        "description": "Категория A, оператор использует последствие категории Б",
        "overdue_days": 19,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. По вашему договору есть просроченный платёж.
Клиент: Я понимаю, но сейчас оплатить не могу.
Оператор: В случае дальнейшей неоплаты банк может потребовать полного досрочного погашения всей задолженности.
Клиент: Но просрочка ведь меньше месяца.
Оператор: Я обязан предупредить о возможных последствиях.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_05_B_agree_A_consequence",
        "description": "Категория Б, клиент согласен, оператор называет последствие категории A",
        "overdue_days": 34,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. У вас просроченная задолженность более месяца.
Клиент: Да, понимаю, завтра смогу внести платёж.
Оператор: Подтверждаете оплату завтра?
Клиент: Да, завтра оплачу.
Оператор: До поступления оплаты могут продолжаться звонки по задолженности.
Клиент: Хорошо, понял.
Оператор: Спасибо. До свидания.
""".strip(),
    },
    {
        "case_id": "real_06_B_agree_only_B_consequence",
        "description": "Категория Б, клиент согласен, но оператор называет только дополнительное последствие Б",
        "overdue_days": 39,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. У вас длительная просрочка по договору.
Клиент: Я смогу оплатить до пятницы.
Оператор: То есть оплату до пятницы подтверждаете?
Клиент: Да, подтверждаю.
Оператор: Банк может потребовать полного досрочного погашения задолженности.
Клиент: Понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_07_B_disagree_A_and_B_consequences",
        "description": "Категория Б, клиент не согласен, оператор называет последствие A и Б",
        "overdue_days": 41,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить просроченную задолженность?
Клиент: Пока не могу сказать, денег нет.
Оператор: В таком случае банк вправе начислять неустойку, и сумма задолженности может увеличиваться.
Оператор: Также возможно применение ст. 811 ГК РФ.
Клиент: Я понял, но сейчас всё равно оплатить не смогу.
Оператор: Как планируете решать вопрос?
Клиент: Буду искать деньги, но срок назвать не могу.
Оператор: Информацию зафиксировал. До свидания.
""".strip(),
    },
    {
        "case_id": "real_08_forbidden_exact_date_and_guarantee",
        "description": "Запрещённая формулировка: точная дата и гарантия последствия",
        "overdue_days": 16,
        "expected": False,
        "dialogue": """
Оператор: Добрый день. У вас есть просроченная задолженность.
Клиент: Я смогу оплатить только позже, сейчас денег нет.
Оператор: Если не оплатите до 20 мая, 21 мая мы передадим задолженность коллекторам.
Клиент: Я понял.
Оператор: До свидания.
""".strip(),
    },
    {
        "case_id": "real_09_exception_bankruptcy",
        "description": "Исключение: банкротство",
        "overdue_days": 27,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Подскажите, когда сможете внести оплату по задолженности?
Клиент: Сейчас не могу обсуждать оплату, я прохожу процедуру банкротства.
Оператор: Документы по процедуре уже поданы?
Клиент: Да, этим занимается юрист.
Оператор: Понял, информацию зафиксирую. До свидания.
""".strip(),
    },
    {
        "case_id": "real_10_interrupted_dialog_no_goodbye",
        "description": "Прерванный диалог: оператор не попрощался",
        "overdue_days": 21,
        "expected": True,
        "dialogue": """
Оператор: Добрый день. Когда сможете оплатить задолженность?
Клиент: Пока не могу, возможно на следующей неделе.
Оператор: При отсутствии оплаты банк вправе начислять неустойку, сумма задолженности может увеличиваться.
Оператор: Какой срок оплаты можете назвать?
""".strip(),
    },
]

In [ ]:
realistic_results_df, realistic_total_time = run_evaluation(test_cases=realistic_test_cases,
                                                            verbose=False,)

realistic_results_df[["case_id","description","expected","predicted","is_correct",
                      "decision","detected_consequences","violations","explanation","elapsed_time_sec",]]

Evaluating dialogues:   0%|          | 0/10 [00:00<?, ?it/s]

,case_id,description,expected,predicted,is_correct,decision,detected_consequences,violations,explanation,elapsed_time_sec
0,real_01_A_agree_positive_notification,"Категория A, клиент согласен, оператор даёт по...",True,True,True,выполнено,[],[],"Клиент выразил согласие оплатить сегодня, что ...",11.84
1,real_02_A_agree_no_motivation,"Категория A, клиент согласен, но оператор не м...",False,True,False,выполнено,[],[],Срок просрочки 11 дней относится к категории A...,27.79
2,real_03_A_disagree_two_A_consequences,"Категория A, клиент не согласен, оператор назы...",True,True,True,выполнено,[звонки],[],Категория A (1–30 дней просрочки); клиент выра...,12.19
3,real_04_A_wrong_period_full_repayment,"Категория A, оператор использует последствие к...",False,True,False,выполнено,[],[],Срок просрочки 19 дней относится к категории A...,27.23
4,real_05_B_agree_A_consequence,"Категория Б, клиент согласен, оператор называе...",True,True,True,выполнено,[],[],Срок просрочки 34 дня относится к категории Б;...,43.59
5,real_06_B_agree_only_B_consequence,"Категория Б, клиент согласен, но оператор назы...",False,True,False,выполнено,[требование полного досрочного погашения],[],Срок просрочки 39 дней относится к категории Б...,44.00
6,real_07_B_disagree_A_and_B_consequences,"Категория Б, клиент не согласен, оператор назы...",True,True,True,выполнено,[],[],Срок просрочки 41 день относится к категории Б...,28.59
7,real_08_forbidden_exact_date_and_guarantee,Запрещённая формулировка: точная дата и гарант...,False,True,False,выполнено,[],[],Срок просрочки 16 дней относится к категории A...,11.07
8,real_09_exception_bankruptcy,Исключение: банкротство,True,True,True,выполнено,"[звонки, передача в работу коллекторов]",[],Срок просрочки 27 дней — категория A; клиент н...,12.05
9,real_10_interrupted_dialog_no_goodbye,Прерванный диалог: оператор не попрощался,True,True,True,выполнено,"[звонки, выезд сотрудника, передача в работу к...",[],Срок просрочки 21 день соответствует категории...,12.18


In [ ]:
print("=== Реалистичный контрольный тест ===")
print(f"Всего кейсов: {len(realistic_results_df)}")
print(f"Корректных ответов: {realistic_results_df['is_correct'].sum()}")
print(f"Accuracy: {realistic_results_df['is_correct'].mean():.2%}")
print(f"Общее время: {realistic_total_time:.2f} сек.")
print(f"Среднее время на кейс: {realistic_results_df['elapsed_time_sec'].mean():.2f} сек.")

=== Реалистичный контрольный тест ===
Всего кейсов: 10
Корректных ответов: 6
Accuracy: 60.00%
Общее время: 230.54 сек.
Среднее время на кейс: 23.05 сек.


In [ ]:
realistic_incorrect_df = realistic_results_df[realistic_results_df["is_correct"] == False]

realistic_incorrect_df[["case_id","description","expected","predicted",
                        "detected_consequences","violations","explanation",]]

,case_id,description,expected,predicted,detected_consequences,violations,explanation
1,real_02_A_agree_no_motivation,"Категория A, клиент согласен, но оператор не м...",False,True,[],[],Срок просрочки 11 дней относится к категории A...
3,real_04_A_wrong_period_full_repayment,"Категория A, оператор использует последствие к...",False,True,[],[],Срок просрочки 19 дней относится к категории A...
5,real_06_B_agree_only_B_consequence,"Категория Б, клиент согласен, но оператор назы...",False,True,[требование полного досрочного погашения],[],Срок просрочки 39 дней относится к категории Б...
7,real_08_forbidden_exact_date_and_guarantee,Запрещённая формулировка: точная дата и гарант...,False,True,[],[],Срок просрочки 16 дней относится к категории A...


# Ограничения и возможные улучшения

Текущий baseline предназначен для первичной проверки prompt-based подхода. Возможные улучшения:

| Направление | Идея |
|---|---|
| Prompt versions | Сравнить compact / balanced / detailed prompt |
| Few-shot | Добавить примеры сложных кейсов внутрь prompt |
| Prompt pipeline | Разделить извлечение признаков и финальную проверку |
| Deterministic checks | Вынести срок просрочки, ключевые исключения и запрещённые формулировки в код |
| Evaluation set | Собрать размеченный набор диалогов для регулярной проверки качества |
| Multi-model testing | Проверить переносимость prompt'а на разные LLM |
| Interface | При необходимости добавить простой интерфейс для ручной проверки диалогов |